# Dataset Quickstart

This notebook shows how declarative dataset YAML becomes a runtime dataset that can be loaded, transformed, saved, and forged again.

What you will do:

- load both dict-based and list-based datasets
- preprocess the items with local Python logic
- save the processed datasets back to JSON plus YAML
- reload the saved YAML outputs through the same public `forge(...)` entrypoint


In [ ]:
import sys
from pathlib import Path

NOTEBOOK_DIR = Path.cwd().resolve()
PROJECT_ROOT = next((p for p in [NOTEBOOK_DIR, *NOTEBOOK_DIR.parents] if (p / 'agentic_forge').exists()), NOTEBOOK_DIR)
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

DATASET_DIR = PROJECT_ROOT / 'examples' / 'dataset'
MATERIALS_DIR = DATASET_DIR / 'materials'

from agentic_forge import forge
from agentic_forge.dataset.forged import ForgedItemDataset


## Step 1: Load the source datasets

The two input materials differ only in JSON shape:

- `dict_dataset.yaml` reads a mapping and injects the mapping key into each runtime item
- `list_dataset.yaml` reads a list and uses an existing field as the item key


In [ ]:
dict_dataset: ForgedItemDataset = forge(MATERIALS_DIR / 'dict_dataset.yaml')
list_dataset: ForgedItemDataset = forge(MATERIALS_DIR / 'list_dataset.yaml')

print(dict_dataset[0])
print(list_dataset[0])


## Step 2: Preprocess and save the datasets

The preprocessing functions add normalized fields to each item and then save the results back to disk. The important part is that the saved outputs include new YAML materials, so the processed datasets remain forgeable artifacts instead of becoming ad hoc files.


In [ ]:
def build_list_summary(item: dict) -> str:
    return f"{item['name']} - {item['value']} ({item['category']})"

processed_dict: ForgedItemDataset = dict_dataset.preprocess_and_save(
    json_path=MATERIALS_DIR / 'output_dict.json',
    format='dict',
    input_templates={
        'PROMPT': '{title}: {text}',
        'TOPIC': '{topic}',
    },
    input_prefix='INPUT_',
    raw_prefix='RAW_',
)

processed_list: ForgedItemDataset = list_dataset.preprocess_and_save(
    json_path=MATERIALS_DIR / 'output_list.json',
    format='list',
    key_field='id',
    input_builders={'SUMMARY': build_list_summary},
    input_prefix='INPUT_',
    raw_prefix='SRC_',
)

print(processed_dict.list_input_fields())
print(processed_list.list_input_fields())


## Step 3: Reload the generated materials

This step confirms that the output YAML files are valid first-class materials. If the forge can reload them cleanly, the save path is preserving the same declarative contract.


In [ ]:
reloaded_dict = forge(MATERIALS_DIR / 'output_dict.yaml')
reloaded_list = forge(MATERIALS_DIR / 'output_list.yaml')

print(reloaded_dict[0])
print(reloaded_list[0])


## Step 4: Optional DataLoader support

If `torch` is installed, the forged dataset can expose a DataLoader view. The notebook keeps this section optional so the example still works in a minimal environment.


In [ ]:
try:
    import torch  # noqa: F401
    loader = processed_list.get_dataloader(batch_size=2, shuffle=False)
    batch = next(iter(loader))
    print(batch)
except Exception as exc:
    print('Skipping DataLoader example:', exc)
